# 04_v2 Content Feature Engineering

Create leakage-controlled content features at `membership_row_id` level using only active v2 `Movie_Master` columns: `MOVIE_NUM`, `movie_title`, `ott_release_month`, and `genre`.


In [1]:
import csv
import json
import math
import re
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

PROJECT_ROOT = Path.cwd()
RAW_VIEW = PROJECT_ROOT / "_data" / "01_raw" / "View_History.csv"
RAW_MOVIE = PROJECT_ROOT / "_data" / "01_raw" / "Movie_Master.csv"
STAGE02_DATA_DIR = PROJECT_ROOT / "park.ingyeom" / "reports" / "data" / "02_v2_preprocessing_policy"
MEMBERSHIP_PATH = STAGE02_DATA_DIR / "membership_v2_preprocessed.csv"
USERMAPPING_PATH = STAGE02_DATA_DIR / "usermapping_v2_policy_checked.csv"
STAGE03_DATA_DIR = PROJECT_ROOT / "park.ingyeom" / "reports" / "data" / "03_v2_usage_feature_engineering"
USAGE_W13 = STAGE03_DATA_DIR / "usage_features_v2_w1_3.csv"
USAGE_W14 = STAGE03_DATA_DIR / "usage_features_v2_w1_4.csv"
FEASIBILITY_AUDIT = PROJECT_ROOT / "park.ingyeom" / "reports" / "tables" / "04_v2_content_feature_feasibility" / "04_v2_moviemaster_content_feasibility_audit.csv"
DATA_DIR = PROJECT_ROOT / "park.ingyeom" / "reports" / "data" / "04_v2_content_feature_engineering"
TABLE_DIR = PROJECT_ROOT / "park.ingyeom" / "reports" / "tables" / "04_v2_content_feature_engineering"
DATA_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

WINDOWS = {"w1_3": (0, 20), "w1_4": (0, 27)}
UNAVAILABLE_METADATA_TOKENS = ["country", "rating", "runtime", "wavve", "kobis", "actor", "director", "embedding"]
FORBIDDEN_FEATURE_TOKENS = ["MOVIE_NUM", "movie_title", "end_date", "days_to_end", "days_since_last_watch_to_end"]

def snapshot(paths):
    out = {}
    for path in paths:
        out[str(path)] = {"size": path.stat().st_size, "mtime_ns": path.stat().st_mtime_ns}
    return out

def rel(path):
    return str(path.relative_to(PROJECT_ROOT)).replace("\\", "/")

def read_csv(path):
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        return reader.fieldnames or [], list(reader)

def write_csv(path, rows, fieldnames):
    with path.open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow({field: row.get(field, "") for field in fieldnames})

def write_json(path, payload):
    with path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

def parse_date(value, fmt):
    return datetime.strptime(value, fmt).date()

def parse_release_month(value):
    if value is None or value == "":
        return None
    try:
        raw = int(value)
        year = raw // 100
        month = raw % 100
        if 1900 <= year <= 2100 and 1 <= month <= 12:
            return year * 100 + month
    except Exception:
        return None
    return None

def month_diff(watch_date, release_yyyymm):
    year = release_yyyymm // 100
    month = release_yyyymm % 100
    return (watch_date.year - year) * 12 + (watch_date.month - month)

def safe_float(value):
    try:
        return float(value)
    except Exception:
        return 0.0

def sanitize(value):
    text = value.strip().lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text or "unknown"

raw_before = snapshot([RAW_VIEW, RAW_MOVIE])
stage02_before = snapshot([MEMBERSHIP_PATH, USERMAPPING_PATH])
stage03_before = snapshot([USAGE_W13, USAGE_W14])
feas_before = snapshot([FEASIBILITY_AUDIT])

membership_cols, membership = read_csv(MEMBERSHIP_PATH)
mapping_cols, usermapping = read_csv(USERMAPPING_PATH)
view_cols, view_rows = read_csv(RAW_VIEW)
movie_cols, movie_rows = read_csv(RAW_MOVIE)
usage13_cols, usage13_rows = read_csv(USAGE_W13)
usage14_cols, usage14_rows = read_csv(USAGE_W14)
feas_cols, feas_rows = read_csv(FEASIBILITY_AUDIT)

membership_by_id = {}
membership_ids_by_user_key = defaultdict(list)
for row in membership:
    mid = int(row["membership_row_id"])
    row["_mid"] = mid
    row["_reg_date"] = parse_date(row["reg_date"], "%y-%m-%d")
    membership_by_id[mid] = row
    membership_ids_by_user_key[row["USER_KEY"]].append(mid)

keys_by_user_num = defaultdict(set)
for row in usermapping:
    keys_by_user_num[row["USER_NUM"]].add(row["USER_KEY"])

# 1-4. Deduplicate Movie_Master before content join.
movie_groups = defaultdict(list)
for source_row_number, row in enumerate(movie_rows, start=2):
    stable = dict(row)
    stable["source_row_number"] = source_row_number
    movie_groups[row["MOVIE_NUM"]].append(stable)

deduped_movie = {}
dedup_summary_rows = []
genre_conflict_rows = []
for movie_num in sorted(movie_groups, key=lambda x: int(x) if x.isdigit() else x):
    group = sorted(movie_groups[movie_num], key=lambda r: int(r["source_row_number"]))
    genres = [r.get("genre", "") for r in group]
    non_null_genres = [g for g in genres if g != ""]
    genre_conflict = len(set(non_null_genres)) > 1
    chosen = None
    if non_null_genres:
        for r in group:
            if r.get("genre", "") != "":
                chosen = r
                break
    if chosen is None:
        chosen = group[0]
    deduped_movie[movie_num] = chosen
    conflict_cols = [col for col in movie_cols if len({r.get(col, "") for r in group}) > 1]
    if len(group) > 1:
        dedup_summary_rows.append({
            "MOVIE_NUM": movie_num,
            "duplicate_group_size": len(group),
            "source_row_numbers": "|".join(str(r["source_row_number"]) for r in group),
            "conflicting_columns": "|".join(conflict_cols),
            "genre_conflict_flag": "Y" if genre_conflict else "N",
            "chosen_source_row_number": chosen["source_row_number"],
            "chosen_genre": chosen.get("genre", ""),
            "dedupe_rule": "stable sort by MOVIE_NUM and original row order; choose first non-null genre; preserve conflicts in audit",
        })
    if genre_conflict:
        for r in group:
            genre_conflict_rows.append({
                "MOVIE_NUM": movie_num,
                "source_row_number": r["source_row_number"],
                "movie_title": r.get("movie_title", ""),
                "ott_release_month": r.get("ott_release_month", ""),
                "genre": r.get("genre", ""),
                "chosen_source_row_number": chosen["source_row_number"],
                "chosen_genre": chosen.get("genre", ""),
                "genre_conflict_flag": "Y",
            })

# Expand view logs to retained membership ids and attach deduped movie metadata only.
expanded_logs = []
join_counts = Counter()
temporal_counts = Counter()
coverage_counts = Counter()
release_parse_counts = Counter()
for view_row_id, view in enumerate(view_rows, start=1):
    movie = deduped_movie.get(view["MOVIE_NUM"])
    if movie is None:
        coverage_counts["view_rows_missing_movie_metadata"] += 1
        continue
    coverage_counts["view_rows_with_movie_metadata"] += 1
    release = parse_release_month(movie.get("ott_release_month", ""))
    if release is None:
        release_parse_counts["release_month_parse_failed"] += 1
    else:
        release_parse_counts["release_month_parse_ok"] += 1
    attached_ids = []
    for user_key in keys_by_user_num.get(view["USER_NUM"], set()):
        attached_ids.extend(membership_ids_by_user_key.get(user_key, []))
    join_counts[f"attachment_count_{len(attached_ids)}"] += 1
    watch_date = parse_date(view["watch_day"], "%Y%m%d")
    watch_time = safe_float(view["watch_time(min)"])
    for mid in attached_ids:
        member = membership_by_id[mid]
        rel_day = (watch_date - member["_reg_date"]).days
        if watch_date < member["_reg_date"]:
            temporal_counts["watch_date_lt_reg_date"] += 1
        if rel_day > 27:
            temporal_counts["rel_day_gt_27"] += 1
        expanded_logs.append({
            "membership_row_id": mid,
            "rel_day": rel_day,
            "watch_date": watch_date,
            "watch_time": watch_time,
            "genre": movie.get("genre", ""),
            "release_month": release,
        })

genre_watch_for_major = Counter()
for log in expanded_logs:
    if 0 <= log["rel_day"] <= 27 and log["genre"] != "":
        genre_watch_for_major[log["genre"]] += log["watch_time"]
major_genres = [genre for genre, _ in genre_watch_for_major.most_common(10)]
genre_slug = {genre: sanitize(genre) for genre in major_genres}

def init_stats():
    return {
        "total_watch_time": 0.0,
        "genre_covered_watch_time": 0.0,
        "genre_missing_watch_time": 0.0,
        "genre_watch": Counter(),
        "genre_sessions": Counter(),
        "release_covered_watch_time": 0.0,
        "release_weighted_sum": 0.0,
        "recent_watch_time": 0.0,
        "old_watch_time": 0.0,
    }

stats = {window: {mid: init_stats() for mid in membership_by_id} for window in WINDOWS}
window_temporal = {window: Counter() for window in WINDOWS}
for log in expanded_logs:
    for window, (start, end) in WINDOWS.items():
        if start <= log["rel_day"] <= end:
            s = stats[window][log["membership_row_id"]]
            wt = log["watch_time"]
            s["total_watch_time"] += wt
            if log["genre"]:
                s["genre_covered_watch_time"] += wt
                s["genre_watch"][log["genre"]] += wt
                s["genre_sessions"][log["genre"]] += 1
            else:
                s["genre_missing_watch_time"] += wt
            if log["release_month"] is not None:
                s["release_covered_watch_time"] += wt
                s["release_weighted_sum"] += wt * log["release_month"]
                age_months = month_diff(log["watch_date"], log["release_month"])
                if 0 <= age_months <= 12:
                    s["recent_watch_time"] += wt
                if age_months >= 60:
                    s["old_watch_time"] += wt
            window_temporal[window]["included_logs"] += 1
        else:
            window_temporal[window]["excluded_logs"] += 1

def entropy(counter):
    total = sum(counter.values())
    if total <= 0:
        return 0.0
    value = 0.0
    for count in counter.values():
        p = count / total
        if p > 0:
            value -= p * math.log2(p)
    return value

def build_row(mid, window):
    s = stats[window][mid]
    total = s["total_watch_time"]
    genre_total = s["genre_covered_watch_time"]
    release_total = s["release_covered_watch_time"]
    row = {"membership_row_id": mid}
    def put(name, value):
        row[f"{window}_{name}"] = value
    top_genre = ""
    top_genre_watch = 0.0
    if s["genre_watch"]:
        top_genre, top_genre_watch = s["genre_watch"].most_common(1)[0]
    put("content_has_watch_obs", 1 if total > 0 else 0)
    put("genre_covered_watch_time", round(s["genre_covered_watch_time"], 6))
    put("genre_missing_watch_time", round(s["genre_missing_watch_time"], 6))
    put("genre_covered_watch_ratio", round(s["genre_covered_watch_time"] / total, 6) if total else 0)
    put("genre_missing_watch_ratio", round(s["genre_missing_watch_time"] / total, 6) if total else 0)
    put("genre_unique_count", len(s["genre_watch"]))
    put("top_genre", top_genre)
    put("top_genre_watch_time", round(top_genre_watch, 6))
    put("top_genre_watch_ratio", round(top_genre_watch / genre_total, 6) if genre_total else 0)
    put("genre_entropy", round(entropy(s["genre_watch"]), 6))
    for genre in major_genres:
        slug = genre_slug[genre]
        put(f"genre_ratio_{slug}", round(s["genre_watch"][genre] / genre_total, 6) if genre_total else 0)
        put(f"genre_watch_time_{slug}", round(s["genre_watch"][genre], 6))
        put(f"genre_session_count_{slug}", s["genre_sessions"][genre])
    put("release_month_covered_watch_ratio", round(release_total / total, 6) if total else 0)
    put("avg_ott_release_month_weighted", round(s["release_weighted_sum"] / release_total, 6) if release_total else "")
    put("recent_content_watch_ratio", round(s["recent_watch_time"] / release_total, 6) if release_total else 0)
    put("old_content_watch_ratio", round(s["old_watch_time"] / release_total, 6) if release_total else 0)
    return row

feature_rows = {window: [build_row(mid, window) for mid in sorted(membership_by_id)] for window in WINDOWS}
fieldnames = {}
for window in WINDOWS:
    base = ["membership_row_id", f"{window}_content_has_watch_obs", f"{window}_genre_covered_watch_time", f"{window}_genre_missing_watch_time", f"{window}_genre_covered_watch_ratio", f"{window}_genre_missing_watch_ratio", f"{window}_genre_unique_count", f"{window}_top_genre", f"{window}_top_genre_watch_time", f"{window}_top_genre_watch_ratio", f"{window}_genre_entropy"]
    for genre in major_genres:
        slug = genre_slug[genre]
        base.extend([f"{window}_genre_ratio_{slug}", f"{window}_genre_watch_time_{slug}", f"{window}_genre_session_count_{slug}"])
    base.extend([f"{window}_release_month_covered_watch_ratio", f"{window}_avg_ott_release_month_weighted", f"{window}_recent_content_watch_ratio", f"{window}_old_content_watch_ratio"])
    fieldnames[window] = base
    write_csv(DATA_DIR / f"content_features_v2_{window}.csv", feature_rows[window], base)

dedup_summary = [
    {"metric": "raw_moviemaster_rows", "count": len(movie_rows), "note": "Raw Movie_Master before deduplication."},
    {"metric": "deduplicated_moviemaster_rows", "count": len(deduped_movie), "note": "One representative row per MOVIE_NUM used for content join."},
    {"metric": "duplicate_MOVIE_NUM_groups", "count": sum(1 for g in movie_groups.values() if len(g) > 1), "note": "Preserved in detail rows below."},
    {"metric": "genre_conflict_MOVIE_NUM_groups", "count": len({r["MOVIE_NUM"] for r in genre_conflict_rows}), "note": "Conflicting rows saved separately."},
] + dedup_summary_rows
write_csv(TABLE_DIR / "04_v2_moviemaster_deduplication_summary.csv", dedup_summary, ["metric", "count", "note", "MOVIE_NUM", "duplicate_group_size", "source_row_numbers", "conflicting_columns", "genre_conflict_flag", "chosen_source_row_number", "chosen_genre", "dedupe_rule"])
write_csv(TABLE_DIR / "04_v2_moviemaster_genre_conflict_rows.csv", genre_conflict_rows, ["MOVIE_NUM", "source_row_number", "movie_title", "ott_release_month", "genre", "chosen_source_row_number", "chosen_genre", "genre_conflict_flag"])

join_coverage = [
    {"metric": "raw_View_History_rows", "count": len(view_rows), "note": "Raw view logs."},
    {"metric": "view_rows_with_movie_metadata_after_dedup", "count": coverage_counts["view_rows_with_movie_metadata"], "note": "After deduplicated Movie_Master lookup."},
    {"metric": "view_rows_missing_movie_metadata_after_dedup", "count": coverage_counts["view_rows_missing_movie_metadata"], "note": "Should be zero if all viewed MOVIE_NUM values are covered."},
    {"metric": "temporary_joined_membership_event_rows", "count": len(expanded_logs), "note": "Audit-only expansion before aggregation to membership_row_id."},
]
for metric, count in sorted(join_counts.items()):
    join_coverage.append({"metric": metric, "count": count, "note": "Raw view row attachment count to retained membership_row_id."})
write_csv(TABLE_DIR / "04_v2_content_join_coverage_summary.csv", join_coverage, ["metric", "count", "note"])

temporal_summary = []
for metric, count in temporal_counts.items():
    temporal_summary.append({"scope": "joined_membership_event_level", "metric": metric, "count": count, "note": "Audited only."})
for window, counts in window_temporal.items():
    for metric, count in counts.items():
        temporal_summary.append({"scope": window, "metric": metric, "count": count, "note": "Window inclusion uses watch_date >= reg_date and rel_day bounds."})
write_csv(TABLE_DIR / "04_v2_content_temporal_filter_summary.csv", temporal_summary, ["scope", "metric", "count", "note"])

genre_feature_summary = []
for window in WINDOWS:
    genre_feature_summary.append({"window": window, "metric": "major_genres", "value": "|".join(major_genres), "count": len(major_genres), "note": "Top genres selected by w1_4 watch time among included logs."})
    genre_feature_summary.append({"window": window, "metric": "rows_with_nonempty_top_genre", "value": "", "count": sum(1 for r in feature_rows[window] if r[f"{window}_top_genre"] != ""), "note": "Membership rows with at least one genre-covered watch."})
write_csv(TABLE_DIR / "04_v2_genre_feature_summary.csv", genre_feature_summary, ["window", "metric", "value", "count", "note"])

release_feature_summary = []
for metric, count in release_parse_counts.items():
    release_feature_summary.append({"window": "raw_movie_joined_view_level", "metric": metric, "count": count, "note": "Parse status counted on metadata-covered raw view rows."})
for window in WINDOWS:
    release_feature_summary.append({"window": window, "metric": "rows_with_release_month_covered_watch", "count": sum(1 for r in feature_rows[window] if float(r[f"{window}_release_month_covered_watch_ratio"]) > 0), "note": "Release-month features created because parsing was reliable for used metadata."})
write_csv(TABLE_DIR / "04_v2_release_month_feature_summary.csv", release_feature_summary, ["window", "metric", "count", "note"])

def missing_summary(rows, window):
    out = []
    for col in fieldnames[window]:
        missing = sum(1 for r in rows if r[col] == "")
        out.append({"window": window, "feature": col, "missing_count": missing, "missing_rate": round(missing/len(rows), 6)})
    return out
def numeric_summary(rows, window):
    out = []
    for col in fieldnames[window]:
        if col == "membership_row_id" or col.endswith("top_genre"):
            continue
        vals = []
        for r in rows:
            if r[col] == "":
                continue
            try:
                vals.append(float(r[col]))
            except Exception:
                pass
        if vals:
            out.append({"window": window, "feature": col, "count": len(vals), "min": min(vals), "max": max(vals), "mean": round(sum(vals)/len(vals), 6), "zero_count": sum(1 for v in vals if v == 0)})
    return out
write_csv(TABLE_DIR / "04_v2_content_feature_missing_summary.csv", missing_summary(feature_rows["w1_3"], "w1_3") + missing_summary(feature_rows["w1_4"], "w1_4"), ["window", "feature", "missing_count", "missing_rate"])
write_csv(TABLE_DIR / "04_v2_content_feature_numeric_summary.csv", numeric_summary(feature_rows["w1_3"], "w1_3") + numeric_summary(feature_rows["w1_4"], "w1_4"), ["window", "feature", "count", "min", "max", "mean", "zero_count"])

summary_payload = {
    "scope": "Stage 04 content feature engineering only.",
    "allowed_movie_master_columns": movie_cols,
    "membership_rows": len(membership),
    "content_feature_rows": {window: len(feature_rows[window]) for window in WINDOWS},
    "major_genres": major_genres,
    "deduplication": {"raw_rows": len(movie_rows), "deduped_rows": len(deduped_movie), "genre_conflict_groups": len({r["MOVIE_NUM"] for r in genre_conflict_rows})},
    "data_outputs": [rel(DATA_DIR / "content_features_v2_w1_3.csv"), rel(DATA_DIR / "content_features_v2_w1_4.csv"), rel(DATA_DIR / "content_feature_summary.json")],
}
write_json(DATA_DIR / "content_feature_summary.json", summary_payload)

report_path = DATA_DIR / "04_v2_content_feature_engineering_report.md"
report_lines = [
    "# 04_v2 Content Feature Engineering Report", "",
    "## Scope", "- Created content features only from active v2 `Movie_Master` columns: `MOVIE_NUM`, `movie_title`, `ott_release_month`, `genre`.", "- No unavailable metadata, final modeling dataset, or model training was created.", "",
    "## Movie_Master Deduplication", f"- Raw Movie_Master rows: {len(movie_rows):,}.", f"- Deduplicated MOVIE_NUM rows: {len(deduped_movie):,}.", f"- Genre conflict groups audited: {len({r['MOVIE_NUM'] for r in genre_conflict_rows}):,}.", "- Dedupe rule: stable sort by MOVIE_NUM and original row order, choose first non-null genre, preserve conflicting rows in audit.", "",
    "## Observation Windows", "- `w1_3`: rel_day 0 through 20.", "- `w1_4`: rel_day 0 through 27.", "",
    "## Output Files", f"- {rel(DATA_DIR / 'content_features_v2_w1_3.csv')}", f"- {rel(DATA_DIR / 'content_features_v2_w1_4.csv')}", f"- {rel(DATA_DIR / 'content_feature_summary.json')}", f"- {rel(report_path)}", "",
    "## Notes", "- `MOVIE_NUM` and `movie_title` were used only for joining or audit, not as model features.", "- No end_date-derived content features were created.",
]
report_path.write_text("\n".join(report_lines) + "\n", encoding="utf-8")

raw_after = snapshot([RAW_VIEW, RAW_MOVIE])
stage02_after = snapshot([MEMBERSHIP_PATH, USERMAPPING_PATH])
stage03_after = snapshot([USAGE_W13, USAGE_W14])
feas_after = snapshot([FEASIBILITY_AUDIT])
required_outputs = [DATA_DIR / "content_features_v2_w1_3.csv", DATA_DIR / "content_features_v2_w1_4.csv", DATA_DIR / "content_feature_summary.json", report_path] + [TABLE_DIR / name for name in ["04_v2_moviemaster_deduplication_summary.csv", "04_v2_moviemaster_genre_conflict_rows.csv", "04_v2_content_join_coverage_summary.csv", "04_v2_content_temporal_filter_summary.csv", "04_v2_genre_feature_summary.csv", "04_v2_release_month_feature_summary.csv", "04_v2_content_feature_missing_summary.csv", "04_v2_content_feature_numeric_summary.csv"]]
all_cols = set(fieldnames["w1_3"]) | set(fieldnames["w1_4"])
unavailable_cols = [c for c in all_cols if any(token in c.lower() for token in UNAVAILABLE_METADATA_TOKENS)]
forbidden_cols = [c for c in all_cols if any(token in c for token in FORBIDDEN_FEATURE_TOKENS)]
end_cols = [c for c in all_cols if "end_date" in c or "days_to_end" in c or "days_since_last_watch_to_end" in c]
final_checks = [
    {"check": "raw_files_unchanged", "status": "PASS" if raw_before == raw_after else "FAIL", "detail": "Movie_Master and View_History unchanged"},
    {"check": "no_project_root_data_output_created", "status": "PASS" if not (PROJECT_ROOT / "_data" / "02_interim" / "04_v2_content_feature_engineering").exists() else "FAIL", "detail": "Stage 04 writes only under park.ingyeom/reports"},
    {"check": "raw_duplicated_Movie_Master_not_directly_joined", "status": "PASS" if len(deduped_movie) < len(movie_rows) else "FAIL", "detail": "content join used deduped_movie lookup"},
    {"check": "Movie_Master_deduplicated_before_content_join", "status": "PASS" if len(deduped_movie) == len(movie_groups) else "FAIL", "detail": f"deduped_rows={len(deduped_movie)}"},
    {"check": "genre_conflicts_audited", "status": "PASS" if (TABLE_DIR / "04_v2_moviemaster_genre_conflict_rows.csv").exists() and len(genre_conflict_rows) > 0 else "FAIL", "detail": f"conflict_rows={len(genre_conflict_rows)}"},
    {"check": "one_row_per_membership_row_id_w1_3", "status": "PASS" if len(feature_rows["w1_3"]) == len({r["membership_row_id"] for r in feature_rows["w1_3"]}) == len(membership) else "FAIL", "detail": f"rows={len(feature_rows['w1_3'])}"},
    {"check": "one_row_per_membership_row_id_w1_4", "status": "PASS" if len(feature_rows["w1_4"]) == len({r["membership_row_id"] for r in feature_rows["w1_4"]}) == len(membership) else "FAIL", "detail": f"rows={len(feature_rows['w1_4'])}"},
    {"check": "no_unavailable_metadata_features_created", "status": "PASS" if not unavailable_cols else "FAIL", "detail": "none" if not unavailable_cols else "|".join(unavailable_cols)},
    {"check": "no_join_identifier_or_title_model_features", "status": "PASS" if not forbidden_cols else "FAIL", "detail": "none" if not forbidden_cols else "|".join(forbidden_cols)},
    {"check": "no_end_date_derived_features_created", "status": "PASS" if not end_cols else "FAIL", "detail": "none" if not end_cols else "|".join(end_cols)},
    {"check": "no_model_trained", "status": "PASS", "detail": "No model training code or output"},
    {"check": "stage02_stage03_and_feasibility_inputs_not_overwritten", "status": "PASS" if stage02_before == stage02_after and stage03_before == stage03_after and feas_before == feas_after else "FAIL", "detail": "Input snapshots unchanged"},
    {"check": "all_required_outputs_created", "status": "PASS" if all(p.exists() for p in required_outputs) else "FAIL", "detail": f"required_outputs={len(required_outputs)}"},
]
write_csv(TABLE_DIR / "04_v2_final_checks.csv", final_checks, ["check", "status", "detail"])

print("04_v2 content feature engineering completed.")
for row in final_checks:
    print(f"{row['check']}: {row['status']} - {row['detail']}")


04_v2 content feature engineering completed.
raw_files_unchanged: PASS - Movie_Master and View_History unchanged
no_project_root_data_output_created: PASS - Stage 04 writes only under park.ingyeom/reports
raw_duplicated_Movie_Master_not_directly_joined: PASS - content join used deduped_movie lookup
Movie_Master_deduplicated_before_content_join: PASS - deduped_rows=14018
genre_conflicts_audited: PASS - conflict_rows=32
one_row_per_membership_row_id_w1_3: PASS - rows=23933
one_row_per_membership_row_id_w1_4: PASS - rows=23933
no_unavailable_metadata_features_created: PASS - none
no_join_identifier_or_title_model_features: PASS - none
no_end_date_derived_features_created: PASS - none
no_model_trained: PASS - No model training code or output
stage02_stage03_and_feasibility_inputs_not_overwritten: PASS - Input snapshots unchanged
all_required_outputs_created: PASS - required_outputs=12
